In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [ ]:
# =====================================================================
# HUB COMPOSITION CHECK - run this before naming any Finding after a
# QCEW/NAICS sector
# =====================================================================
# The deck's Findings are built on QCEW Sector Title - a NAICS-derived
# INDUSTRY classification, independent of Program Hub Category (our own
# 4-hub internal labeling). These are two different axes that can and do
# diverge - a "Health Care and Social Assistance" sector finding can be
# majority-driven by Community and Social Services hub placements, not
# our Healthcare hub, simply because NAICS 62 includes real subindustries
# (Individual and Family Services, Child Day Care, etc.) that are
# colloquially "social services," not clinical healthcare.
#
# This script exists so that gap gets caught automatically before a
# slide gets named after it, not discovered after the fact. Run it
# against any QCEW sector before writing a Finding about it.

import pandas as pd

CODED_PATH = data_path("AllRoles_nioccs_coded_with_education.csv")

QCEW_SECTOR_TITLES = {
    "11": "Agriculture, Forestry, Fishing and Hunting", "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities", "23": "Construction", "31-33": "Manufacturing", "42": "Wholesale Trade",
    "44-45": "Retail Trade", "48-49": "Transportation and Warehousing", "51": "Information",
    "52": "Finance and Insurance", "53": "Real Estate and Rental and Leasing",
    "54": "Professional and Technical Services", "55": "Management of Companies and Enterprises",
    "56": "Administrative and Waste Services", "61": "Educational Services",
    "62": "Health Care and Social Assistance", "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services", "81": "Other Services (except Public Administration)",
    "92": "Public Administration", "99": "Unclassified",
}

# Flag any sector where a SINGLE hub other than the "expected" one (if the
# sector's name suggests a specific hub) drives more than this share -
# worth a second look before naming a Finding after it.
DOMINANCE_FLAG_THRESHOLD = 50.0


def naics_to_qcew_sector(naics_code):
    if pd.isna(naics_code):
        return None
    code = str(naics_code).strip()
    prefix2 = code[:2]
    if prefix2 in ("31", "32", "33"):
        return "31-33"
    if prefix2 in ("44", "45"):
        return "44-45"
    if prefix2 in ("48", "49"):
        return "48-49"
    return prefix2


coded = pd.read_csv(CODED_PATH, encoding="utf-8-sig")
merged = coded[coded["API Call Error"].isna()].copy()
merged["QCEW Sector Title"] = merged["NAICS Code"].apply(naics_to_qcew_sector).map(QCEW_SECTOR_TITLES)

print("=" * 78)
print("Hub composition behind every QCEW sector with meaningful volume")
print("=" * 78)

sector_counts = merged["QCEW Sector Title"].value_counts()
for sector in sector_counts.index:
    n = sector_counts[sector]
    if n < 20:  # skip near-empty sectors, not enough to say anything meaningful
        continue
    sub = merged[merged["QCEW Sector Title"] == sector]
    composition = (sub["Program Hub Category"].value_counts(normalize=True) * 100).round(1)
    top_hub = composition.index[0]
    top_share = composition.iloc[0]

    flag = ""
    # A sector whose name closely matches one of our hub names, but whose
    # placements are majority-driven by a DIFFERENT hub, is exactly the
    # trap this script exists to catch. Spaces stripped before comparing -
    # "Healthcare" vs "Health Care" is exactly the kind of near-miss that
    # slips past a naive substring check.
    sector_normalized = sector.lower().replace(" ", "")
    expected_hub = None
    for hub in merged["Program Hub Category"].dropna().unique():
        hub_key = hub.split()[0].lower().replace(" ", "")
        if hub_key in sector_normalized:
            expected_hub = hub
            break
    if expected_hub and top_hub != expected_hub:
        flag = f"  <-- CHECK: sector name resembles '{expected_hub}', but '{top_hub}' actually dominates"

    print(f"\n{sector} (n={n}){flag}")
    print(composition.to_string())

print("\n" + "=" * 78)
print("Before naming a Finding after any sector above, confirm the framing "
      "credits the actual dominant hub(s), not just the sector's name.")